# Brain MRI Segmentation — Training on Colab

**Pré-requis :** le stack Docker tourne sur ton PC local (`docker compose up -d --build`) et tu as récupéré l'URL ngrok depuis http://localhost:4040

## 0. Configuration

In [ ]:
# ← Mettre à jour à chaque redémarrage du stack Docker
TRACKING_URI = "https://xxxx.ngrok-free.app"

# Paramètres d'entraînement
UNET_EPOCHS   = 30
UNET_ENCODER  = "resnet34"
UNET_BATCH    = 16
UNET_IMG_SIZE = 256

YOLO_EPOCHS   = 10
YOLO_BATCH    = 16
YOLO_IMG_SIZE = 256

YOLO_DATASET_DIR = "/content/yolo_dataset"

## 1. Setup environnement

In [ ]:
!git clone https://github.com/RomeoCorrec/Brain-MRI-Segmentation.git
%cd Brain-MRI-Segmentation

In [ ]:
# torch et torchvision sont déjà présents sur Colab
!pip install -q \
    segmentation-models-pytorch==0.5.0 \
    ultralytics==8.3.239 \
    albumentations==2.0.8 \
    mlflow==2.14.3 \
    boto3==1.38.0

# Réinstaller numpy pour corriger les incompatibilités binaires avec pandas
!pip install -q -U numpy

# ⚠️ Redémarrer le runtime après cette cellule : Runtime > Restart session
print("\n⚠️  Redémarre le runtime maintenant (Runtime > Restart session), puis continue à la cellule suivante.")

In [ ]:
# Reprendre ici après le redémarrage du runtime
%cd /content/Brain-MRI-Segmentation

import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## 2. Télécharger le dataset TCGA (Kaggle)

In [ ]:
!pip install -q kaggle

# Uploader ton fichier kaggle.json
# (téléchargeable sur https://www.kaggle.com/settings → API → Create New Token)
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d mateuszbuda/lgg-mri-segmentation --unzip -p /content/data
!ls /content/data/

In [ ]:
import glob
masks = glob.glob("/content/data/kaggle_3m/*/*_mask.tif")
print(f"{len(masks)} masques trouvés")

DATA_DIR = "/content/data/kaggle_3m"

## 3. Vérifier la connexion MLFlow

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiments = [e.name for e in client.search_experiments()]
print("Expériences MLFlow :", experiments)
# Attendu : ['Default']  (vide au premier lancement)

## 4. Entraînement UNet

In [ ]:
!python src/train_unet.py \
    --tracking-uri {TRACKING_URI} \
    --data-dir {DATA_DIR} \
    --encoder {UNET_ENCODER} \
    --epochs {UNET_EPOCHS} \
    --batch-size {UNET_BATCH} \
    --image-size {UNET_IMG_SIZE} \
    --output-dir /content/outputs

## 5. Entraînement YOLOv8

In [ ]:
# Convertir les masques TCGA au format YOLO segmentation (polygones normalisés)
!python src/prepare_yolo_dataset.py \
    --data-dir {DATA_DIR} \
    --output-dir {YOLO_DATASET_DIR}

In [ ]:
YOLO_YAML = f"{YOLO_DATASET_DIR}/brain_tumor.yaml"

!python src/train_yolo.py \
    --tracking-uri {TRACKING_URI} \
    --data {YOLO_YAML} \
    --epochs {YOLO_EPOCHS} \
    --batch {YOLO_BATCH} \
    --imgsz {YOLO_IMG_SIZE}

## 6. Vérifier les résultats

In [ ]:
import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

print("=== Expériences ===")
for exp in client.search_experiments():
    print(f"  {exp.name}")

print("\n=== Modèles enregistrés ===")
for model in client.search_registered_models():
    latest = model.latest_versions
    for v in latest:
        print(f"  {model.name} — v{v.version} ({v.current_stage})")

## 7. Comparaison UNet vs YOLOv8

Cette section compare les deux modèles sur trois axes :
- **Courbes d'entraînement** (loss + métriques par epoch)
- **Métriques finales** (tableau récapitulatif)
- **Prédictions visuelles** sur des images de validation

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()


def find_training_run(experiment_name):
    """Return the run with the most metrics logged (skips utility runs like model-registration)."""
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        print(f"Expérience '{experiment_name}' non trouvée.")
        return None
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["start_time DESC"],
    )
    # Prendre le run avec le plus de métriques loggées
    best = max(runs, key=lambda r: len(r.data.metrics), default=None)
    return best


def get_history(run, metric_names):
    history = {}
    for name in metric_names:
        records = client.get_metric_history(run.info.run_id, name)
        if records:
            history[name] = [r.value for r in sorted(records, key=lambda r: r.step)]
    return history


unet_run = find_training_run("UNet-Segmentation")
yolo_run = find_training_run("YOLOv8-Segmentation")

# Inspecter les métriques disponibles
if unet_run:
    print("UNet métriques disponibles :", list(unet_run.data.metrics.keys()))
if yolo_run:
    print("YOLO métriques disponibles :", list(yolo_run.data.metrics.keys()))

In [ ]:
# Récupérer l'historique par epoch
unet_h = get_history(unet_run, ["train_loss", "val_loss", "val_dice"]) if unet_run else {}

yolo_metric_candidates = [
    "train/seg_loss", "val/seg_loss",
    "metrics/mAP50M", "metrics/mAP50-95M",
    "metrics/precisionM", "metrics/recallM",
]
yolo_h = get_history(yolo_run, yolo_metric_candidates) if yolo_run else {}

print("UNet  historique :", list(unet_h.keys()))
print("YOLO  historique :", list(yolo_h.keys()))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Courbes d'entraînement — UNet vs YOLOv8", fontsize=14, fontweight="bold")

# --- UNet ---
ax = axes[0, 0]
if "train_loss" in unet_h and "val_loss" in unet_h:
    ax.plot(unet_h["train_loss"], label="train", color="steelblue")
    ax.plot(unet_h["val_loss"],   label="val",   color="orange")
ax.set_title("UNet — Loss (BCE)")
ax.set_xlabel("Epoch")
ax.legend()

ax = axes[0, 1]
if "val_dice" in unet_h:
    ax.plot(unet_h["val_dice"], color="green", label="val dice")
ax.set_title("UNet — Dice Score (val)")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
ax.legend()

axes[0, 2].axis("off")

# --- YOLO ---
ax = axes[1, 0]
if "train/seg_loss" in yolo_h and "val/seg_loss" in yolo_h:
    ax.plot(yolo_h["train/seg_loss"], label="train", color="steelblue")
    ax.plot(yolo_h["val/seg_loss"],   label="val",   color="orange")
ax.set_title("YOLOv8 — Seg Loss")
ax.set_xlabel("Epoch")
ax.legend()

ax = axes[1, 1]
if "metrics/mAP50M" in yolo_h:
    ax.plot(yolo_h["metrics/mAP50M"], color="purple", label="mAP50")
if "metrics/mAP50-95M" in yolo_h:
    ax.plot(yolo_h["metrics/mAP50-95M"], color="violet", linestyle="--", label="mAP50-95")
ax.set_title("YOLOv8 — mAP (val)")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
ax.legend()

ax = axes[1, 2]
if "metrics/precisionM" in yolo_h and "metrics/recallM" in yolo_h:
    ax.plot(yolo_h["metrics/precisionM"], color="teal",   label="precision")
    ax.plot(yolo_h["metrics/recallM"],    color="salmon", label="recall")
ax.set_title("YOLOv8 — Precision / Recall")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig("/content/comparison_curves.png", dpi=150)
plt.show()
print("Sauvegardé : /content/comparison_curves.png")

In [ ]:
# Tableau récapitulatif des métriques finales
rows = []

if unet_run:
    m = unet_run.data.metrics
    rows.append({
        "Modèle": "UNet (resnet34)",
        "Val Loss": f"{m.get('val_loss', float('nan')):.4f}",
        "Val Dice": f"{m.get('val_dice', float('nan')):.4f}",
        "mAP50": "—",
        "mAP50-95": "—",
    })

if yolo_run:
    m = yolo_run.data.metrics
    rows.append({
        "Modèle": "YOLOv8n-seg",
        "Val Loss": f"{m.get('val/seg_loss', float('nan')):.4f}",
        "Val Dice": "—",
        "mAP50": f"{m.get('metrics/mAP50M', float('nan')):.4f}",
        "mAP50-95": f"{m.get('metrics/mAP50-95M', float('nan')):.4f}",
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

In [ ]:
# Prédictions visuelles sur des images de validation
import sys, os, glob, random, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from ultralytics import YOLO

sys.path.insert(0, "/content/Brain-MRI-Segmentation")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 256
N_SAMPLES = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Charger UNet ---
unet_model = smp.Unet(
    encoder_name=UNET_ENCODER,
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)
unet_model.load_state_dict(torch.load("/content/outputs/best_unet.pth", map_location=DEVICE))
unet_model.eval()

# --- Charger YOLOv8 ---
yolo_weights = sorted(glob.glob("/content/runs/mri_yolo_experiment*/weights/best.pt"))[-1]
yolo_model = YOLO(yolo_weights)
print(f"UNet chargé | YOLO chargé depuis {yolo_weights}")

# --- Transforms UNet ---
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# --- Sélectionner des images de validation ---
val_imgs = sorted(glob.glob(f"{YOLO_DATASET_DIR}/images/val/*.jpg"))
val_labels = [p.replace("/images/", "/labels/").replace(".jpg", ".txt") for p in val_imgs]
samples = random.sample(list(zip(val_imgs, val_labels)), min(N_SAMPLES, len(val_imgs)))

fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4 * len(samples)))
fig.suptitle("Comparaison visuelle — UNet vs YOLOv8", fontsize=13, fontweight="bold")
col_titles = ["Image originale", "Vérité terrain", "UNet", "YOLOv8"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11)

for row, (img_path, lbl_path) in enumerate(samples):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    # Ground truth (polygones YOLO → masque binaire)
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 7:
                    coords = list(map(float, parts[1:]))
                    pts = np.array([[coords[i]*w, coords[i+1]*h]
                                    for i in range(0, len(coords), 2)], dtype=np.int32)
                    cv2.fillPoly(gt_mask, [pts], 255)

    # Prédiction UNet
    aug = val_tf(image=img_rgb)
    tensor = aug["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logit = unet_model(tensor)
    unet_mask = (torch.sigmoid(logit).squeeze().cpu().numpy() > 0.5).astype(np.uint8) * 255
    unet_mask = cv2.resize(unet_mask, (w, h))

    # Prédiction YOLO
    results = yolo_model.predict(img_path, imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    yolo_mask = np.zeros((h, w), dtype=np.uint8)
    if results.masks is not None:
        for mask_data in results.masks.data:
            m = mask_data.cpu().numpy()
            m = cv2.resize(m, (w, h))
            yolo_mask = np.maximum(yolo_mask, (m > 0.5).astype(np.uint8) * 255)

    for col, img_data in enumerate([img_rgb, gt_mask, unet_mask, yolo_mask]):
        ax = axes[row, col]
        ax.imshow(img_data, cmap=None if col == 0 else "gray", vmin=None if col == 0 else 0, vmax=None if col == 0 else 255)
        ax.axis("off")

plt.tight_layout()
plt.savefig("/content/comparison_predictions.png", dpi=150)
plt.show()
print("Sauvegardé : /content/comparison_predictions.png")